# BS4 실패 URL Selenium 재시도 (로컬용)

`본문_bs4_재실패_*.json`에 들어 있는 실패 URL만 Selenium으로 다시 열어 본다. 성공한 기사는 별도 CSV로 저장하고, 필요하면 기존 `본문_bs4_*.csv`에 병합한다.

- 입력: `data/본문_bs4_재실패_*.json`
- 출력: `data/본문_selenium_재시도성공_*.csv`, `data/본문_selenium_재시도실패_*.json`
- 특징: 실패 URL만 재시도, 브라우저 화면 확인 가능, 기존 CSV 백업 후 병합


In [ ]:
# 필요한 패키지 설치
# 로컬 커널에 selenium/pandas가 이미 있으면 이 셀은 건너뛰어도 됨
%pip install -q selenium pandas


In [ ]:
from pathlib import Path
import json
import os
import platform
import random
import shutil
import subprocess
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 로컬/Colab 결과 차이를 줄이기 위해 URL 수집 때와 같은 User-Agent 사용
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 브라우저를 직접 보면서 확인하려면 False 유지, 창 없이 돌리려면 True로 변경
HEADLESS = False

# 재시도할 때 서버에 너무 촘촘히 붙지 않도록 대기
RETRY_PAUSE_RANGE_SEC = (1.0, 2.5)
PAGE_LOAD_WAIT_SEC = 0.8
SELENIUM_WAIT_SEC = 8

# 성공분을 기존 본문_bs4 CSV에 바로 합치기
MERGE_TO_BS4_CSV = True

# 병합 전 기존 CSV 백업 만들기
MAKE_BACKUP = True

# None이면 data 폴더의 모든 본문_bs4_재실패_*.json 재시도
# 특정 파일만 하려면 예: TARGET_FAILURE_FILES = ['본문_bs4_재실패_KT_250901_250930.json']
TARGET_FAILURE_FILES = None

# 현재 노트북 위치 기준 data 폴더를 기본값으로 사용
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'crawling':
    candidate = Path('/home/carol/Text-data-Analysis_26-Spring/notebook/crawling')
    NOTEBOOK_DIR = candidate if candidate.exists() else NOTEBOOK_DIR

DATA_DIR = NOTEBOOK_DIR / 'data'
print(f'DATA_DIR: {DATA_DIR}')
print(f'User-Agent: {USER_AGENT}')


In [ ]:
# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range=RETRY_PAUSE_RANGE_SEC):
    pause_sec = random.uniform(*pause_range)
    print(f'{label} {pause_sec:.1f}초 대기')
    time.sleep(pause_sec)


# 로컬 OS에 설치된 Chrome 실행 파일 탐색
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)

    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# Selenium 드라이버 생성
# 실패 URL을 눈으로 확인할 수 있도록 기본값은 브라우저 창을 띄우는 방식
def build_driver(headless=HEADLESS):
    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 수집 환경 고정
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동화 제어 관련 switch 제외
    options.add_experimental_option('useAutomationExtension', False)  # Selenium 자동화 확장 비활성화
    options.add_argument('--disable-blink-features=AutomationControlled')  # AutomationControlled 플래그 비활성화
    options.add_argument('--window-size=1400,1000')  # 일정한 화면 크기로 렌더링

    if headless:
        options.add_argument('--headless=new')  # 창 없이 실행할 때 새 headless 모드 사용

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스 컨테이너/WSL 환경에서 Chrome 실행 안정화
        options.add_argument('--disable-dev-shm-usage')  # /dev/shm 용량 부족으로 Chrome이 죽는 문제 완화
        options.add_argument('--disable-gpu')  # headless 환경에서 GPU 관련 오류 방지

    # Chrome 실행 파일을 찾으면 명시하고, 못 찾으면 Selenium Manager 기본 탐색 사용
    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'Chrome binary: {chrome_binary}')
        subprocess.run([chrome_binary, '--version'], check=False)
    else:
        print('Chrome binary를 직접 찾지 못함 — Selenium Manager 기본 탐색 사용')

    # Selenium Manager가 현재 Chrome 버전에 맞는 ChromeDriver를 자동으로 찾거나 내려받음
    driver = webdriver.Chrome(service=Service(), options=options)

    # navigator.webdriver 플래그 제거
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


driver = build_driver()


In [ ]:
def extract_article_selenium(driver, link):
    # 실제 네이버 뉴스 웹페이지로 이동
    driver.get(link)

    # 페이지 로딩 대기
    wait = WebDriverWait(driver, SELENIUM_WAIT_SEC)
    time.sleep(PAGE_LOAD_WAIT_SEC)

    # 제목 추출하기
    title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
    title = title_elements[0].text.strip() if title_elements else ''

    # 본문 추출하기
    body_elements = driver.find_elements(By.ID, 'newsct_article')
    body = body_elements[0].text.replace('\n', '').strip() if body_elements else ''

    # 날짜 추출하기
    pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else ''

    # 카테고리 추출하기
    category_elements = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
    category = category_elements[0].text.strip() if category_elements else ''

    if not title or not body or not pubdate:
        # 아주 짧게 한 번 더 기다린 뒤 재확인
        wait.until(lambda d: d.find_elements(By.ID, 'newsct_article') or d.find_elements(By.CLASS_NAME, 'media_end_head_headline'))
        title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        body_elements = driver.find_elements(By.ID, 'newsct_article')
        pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')

        title = title_elements[0].text.strip() if title_elements else title
        body = body_elements[0].text.replace('\n', '').strip() if body_elements else body
        pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else pubdate

    # 제목/본문/날짜 중 하나라도 없으면 실패로 기록
    if not title or not body or not pubdate:
        raise ValueError(f'title/body/pubdate 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


In [ ]:
# BS4 수집에서 재실패한 JSON 파일 불러오기
def load_failure_files(data_dir=DATA_DIR):
    # TARGET_FAILURE_FILES가 지정되면 해당 파일만, 아니면 모든 재실패 파일 처리
    if TARGET_FAILURE_FILES:
        paths = [data_dir / name for name in TARGET_FAILURE_FILES]
    else:
        paths = sorted(data_dir.glob('본문_bs4_재실패_*.json'))

    existing = [p for p in paths if p.exists()]
    missing = [p for p in paths if not p.exists()]

    if missing:
        print('찾지 못한 실패 파일:')
        for p in missing:
            print(f'- {p}')

    if not existing:
        raise FileNotFoundError('재시도할 본문_bs4_재실패_*.json 파일이 없습니다.')

    print(f'재시도 대상 실패 파일: {len(existing)}개')
    for p in existing:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'- {p.name}: {len(data.get("links", []))}건')
    return existing


failure_files = load_failure_files()


In [ ]:
# 실패 URL만 Selenium으로 다시 방문해 본문 수집 재시도
def retry_failure_file(failure_path, driver=driver, data_dir=DATA_DIR):
    stem = failure_path.stem.replace('본문_bs4_재실패_', '')
    bs4_csv_path = data_dir / f'본문_bs4_{stem}.csv'
    success_path = data_dir / f'본문_selenium_재시도성공_{stem}.csv'
    remain_fail_path = data_dir / f'본문_selenium_재시도실패_{stem}.json'

    # 재실패 JSON에서 원래 index와 URL 목록 불러오기
    with failure_path.open('r', encoding='utf-8') as f:
        failure_data = json.load(f)

    failed_indices = failure_data.get('err_idx', [])
    failed_links = failure_data.get('links', [])

    successes = []
    failures = []

    print()
    print(f'=== {stem} Selenium 재시도 시작: {len(failed_links)}건 ===')

    # 실패 URL을 하나씩 Selenium으로 재방문
    for seq, (original_idx, link) in enumerate(zip(failed_indices, failed_links), start=1):
        try:
            article = extract_article_selenium(driver, link)
            article['original_idx'] = original_idx
            successes.append(article)
            print(f'성공 [{seq}/{len(failed_links)}] index={original_idx}')
        except Exception as exc:
            failures.append({'original_idx': original_idx, 'link': link, 'error': repr(exc)})
            print(f'실패 [{seq}/{len(failed_links)}] index={original_idx}: {exc!r}')

        polite_sleep('다음 재시도 전')

    # Selenium 재시도 성공분 저장
    success_df = pd.DataFrame(successes)
    if not success_df.empty:
        success_df.to_csv(success_path, index=False, encoding='utf-8-sig')
        print(f'Selenium 성공분 저장: {success_path}')

    # 재시도 후에도 실패한 URL과 오류 이유 저장
    with remain_fail_path.open('w', encoding='utf-8') as f:
        json.dump({'failures': failures}, f, ensure_ascii=False, indent=2)
    print(f'Selenium 재시도 실패 목록 저장: {remain_fail_path}')

    # 성공분을 기존 BS4 CSV에 병합
    if MERGE_TO_BS4_CSV and not success_df.empty:
        if not bs4_csv_path.exists():
            raise FileNotFoundError(f'병합할 기존 CSV가 없습니다: {bs4_csv_path}')

        # 기존 CSV와 성공분을 합친 뒤 링크 기준 중복 제거, 날짜순 정렬
        original_df = pd.read_csv(bs4_csv_path, encoding='utf-8-sig')
        merge_df = success_df[['link', 'pubdate', 'category', 'title', 'body']].copy()
        merged_df = pd.concat([original_df, merge_df], ignore_index=True)
        merged_df = merged_df.drop_duplicates(subset=['link'], keep='last').reset_index(drop=True)
        merged_df['pubdate'] = pd.to_datetime(merged_df['pubdate'], errors='coerce')
        merged_df = merged_df.sort_values('pubdate').reset_index(drop=True)

        # 병합 전 기존 CSV 백업
        if MAKE_BACKUP:
            backup_path = bs4_csv_path.with_suffix('.csv.bak_before_selenium_retry')
            if not backup_path.exists():
                shutil.copy2(bs4_csv_path, backup_path)
                print(f'기존 CSV 백업: {backup_path}')

        merged_df.to_csv(bs4_csv_path, index=False, encoding='utf-8-sig')
        print(f'기존 CSV 병합 완료: {bs4_csv_path}')
        print(f'행 수: {len(original_df)} -> {len(merged_df)}')

    print(f'완료: 성공 {len(successes)}건 / 실패 {len(failures)}건')
    return {
        'period': stem,
        'success': len(successes),
        'fail': len(failures),
        'success_path': str(success_path),
        'remain_fail_path': str(remain_fail_path),
    }


In [ ]:
# 생성된 failure_files를 순서대로 재시도
# 한 파일이 실패해도 결과를 요약할 수 있게 파일별 결과 저장
retry_results = []
for failure_path in failure_files:
    result = retry_failure_file(failure_path)
    retry_results.append(result)

summary_df = pd.DataFrame(retry_results)
summary_df


In [ ]:
# 작업이 끝나면 브라우저 종료
# 필요하면 이 셀만 따로 실행
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')
